In [0]:
use catalog wh_gb_test;

use schema gb_test;

drop table if exists dim_customer;

drop table if exists stg_customer;
-- scd 2 table 
CREATE TABLE dim_customer (
    sk BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    customer_id INT,
    customer_name VARCHAR(100),
    city VARCHAR(100),
    start_date DATE,
    end_date DATE,
    is_current CHAR(1)
);


-- source 
CREATE TABLE stg_customer (
    customer_id INT,
    customer_name VARCHAR(100),
    city VARCHAR(100)
);


-- day 1 

INSERT INTO stg_customer VALUES
(101,'Ravi','Chennai'),
(102,'Arun','Bangalore'),
(103,'Priya','Hyderabad'),
(104,'Karthik','Mumbai'),
(105,'Divya','Pune'),
(106,'Suresh','Delhi'),
(107,'Meena','Kolkata'),
(108,'Vijay','Chennai'),
(109,'Anita','Bangalore'),
(110,'Rahul','Hyderabad');


-- day 1 load -> staging to dim

INSERT INTO dim_customer
(customer_id,customer_name,city,start_date,end_date,is_current)
SELECT
customer_id,
customer_name,
city,
'2025-01-01',
'9999-12-31',
'Y'
FROM stg_customer;

-- day 2 -> updatded , newly added (incremental )
-- overwrite with day2 data in staging table 

TRUNCATE TABLE stg_customer;

INSERT INTO stg_customer VALUES
(101,'Ravi','Bangalore'),
(102,'Arun','Bangalore'),
(103,'Priya','Chennai'),
(110,'Rahul','Mumbai'),
(111,'Kiran','Chennai');

-- upsert
-- updated customer -> make the old record as inactive
--                 -> insert a new record 

-- new customer  ->  insert ;

select * from stg_customer;

select * from dim_customer;

 


UPDATE dim_customer
SET
    end_date   = '2025-01-02',
    is_current = 'N'
WHERE is_current = 'Y'
  AND customer_id IN (
      SELECT s.customer_id
      FROM stg_customer s
      INNER JOIN dim_customer d
          ON d.customer_id = s.customer_id
      WHERE d.city <> s.city
        AND d.is_current = 'Y'
  );

INSERT INTO dim_customer
(customer_id,customer_name,city,start_date,end_date,is_current)
SELECT
s.customer_id,
s.customer_name,
s.city,
'2025-01-02',
'9999-12-31',
'Y'
FROM stg_customer s
JOIN dim_customer d
ON s.customer_id = d.customer_id
WHERE d.is_current='N'
AND d.end_date='2025-01-02';


INSERT INTO dim_customer
(customer_id,customer_name,city,start_date,end_date,is_current)
SELECT
s.customer_id,
s.customer_name,
s.city,
'2025-01-02',
'9999-12-31',
'Y'
FROM stg_customer s
LEFT JOIN dim_customer d
ON s.customer_id = d.customer_id
WHERE d.customer_id IS NULL;


select * from dim_customer;




